In [1]:
"""
Input:
["hello world",
 "cat sat on mat"]

Vocab ->  token IDs
PAD("") ->0, UNK("[UNK]")->>1
“hello”→2, “world”→3, “cat”→4, “sat”→5, “on”→6, “mat”→7

3. Padded token-ID sequence (to maxlen=5)
t0 t1 t2 t3 t4
2   3  1  0   0
4.  5. 6. 7.  0

4. Embedding matrix shape (8*3) maps each ID -> a 3-dim vector
[[ 1.76405235  0.40015721  0.97873798] #pad
[ 2.2408932   1.86755799 -0.97727788] #unk
[ 0.95008842 -0.15135721 -0.10321885] #hello
[ 0.4105985   0.14404357  1.45427351] #world
[ 0.76103773  0.12167502  0.44386323] #cat
[ 0.33367433  1.49407907 -0.20515826] #sat
[ 0.3130677  -0.85409574 -2.55298982] #on
[ 0.6536186   0.8644362  -0.74216502]] #mat

5. Embedded sequence shape(2,5,3):
[
		[
    			[ 0.95, -0.15, -0.10], ← "hello" 
			[ 0.41,  0.14,  1.45], ← "world" 
			[ 1.76,  0.40,  0.98], ← UNK 
			[ 1.76,  0.40,  0.98], ← PAD
			[ 1.76,  0.40,  0.98]  <- PAD
		] 

		 [
			[ 0.76,  0.12,  0.44],   ← "cat"
  			[ 0.33,  1.49, -0.21],   ← "sat"
  			[ 0.31, -0.85, -2.55],   ← "on"
  			[ 0.65,  0.86, -0.74],   ← "mat"
  			[ 1.76,  0.40,  0.98]     <- PAD
		]

]  
"""

'\nInput:\n["hello world",\n "cat sat on mat"]\n\nVocab ->  token IDs\nPAD("") ->0, UNK("[UNK]")->>1\n“hello”→2, “world”→3, “cat”→4, “sat”→5, “on”→6, “mat”→7\n\n3. Padded token-ID sequence (to maxlen=5)\nt0 t1 t2 t3 t4\n2   3  1  0   0\n4.  5. 6. 7.  0\n\n4. Embedding matrix shape (8*3) maps each ID -> a 3-dim vector\n[[ 1.76405235  0.40015721  0.97873798] #pad\n[ 2.2408932   1.86755799 -0.97727788] #unk\n[ 0.95008842 -0.15135721 -0.10321885] #hello\n[ 0.4105985   0.14404357  1.45427351] #world\n[ 0.76103773  0.12167502  0.44386323] #cat\n[ 0.33367433  1.49407907 -0.20515826] #sat\n[ 0.3130677  -0.85409574 -2.55298982] #on\n[ 0.6536186   0.8644362  -0.74216502]] #mat\n\n5. Embedded sequence shape(2,5,3):\n[\n\t\t[\n    \t\t\t[ 0.95, -0.15, -0.10], ← "hello" \n\t\t\t[ 0.41,  0.14,  1.45], ← "world" \n\t\t\t[ 1.76,  0.40,  0.98], ← UNK \n\t\t\t[ 1.76,  0.40,  0.98], ← PAD\n\t\t\t[ 1.76,  0.40,  0.98]  <- PAD\n\t\t] \n\n\t\t [\n\t\t\t[ 0.76,  0.12,  0.44],   ← "cat"\n  \t\t\t[ 0.33,  1.49

In [2]:
import pandas as pd
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, Model, callbacks
from sklearn.model_selection import train_test_split
from tqdm import tqdm

In [3]:
# ────────────────────────────────────────────────────────────────────────────────
# 0) Hyperparameters & Constants
# ────────────────────────────────────────────────────────────────────────────────
MAX_VOCAB_SIZE    = 20000
MAX_SEQUENCE_LEN  = 200
EMBEDDING_DIM     = 300
LSTM_UNITS        = 64
BATCH_SIZE        = 64
EPOCHS            = 1
AUTOTUNE          = tf.data.AUTOTUNE
NUM_CLASSES       = 4
CLASS_NAMES       = ["World", "Sports", "Business", "Sci/Tech"]


In [4]:
train_path = "/Users/sameerkhan/Desktop/sameerkhan/data/nlp/ag_news/train.csv"
test_path = "/Users/sameerkhan/Desktop/sameerkhan/data/nlp/ag_news/test.csv"

MODEL_DIR = "/Users/sameerkhan/Desktop/sameerkhan/weights/nlp/bilstm"
os.makedirs(MODEL_DIR, exist_ok=True)

CHECKPOINT_FILE = f"agnews_bilstm_fun_glove_1.h5"

FINAL_MODEL_FILE = "agnews_bilstm_fun_glove_1.keras"

VOCAB_FILE = MODEL_DIR + "/agnews_vocab.txt"

GLOVE_FILE = '/Users/sameerkhan/Desktop/sameerkhan/data/nlp/glove.42B.300d.txt'

In [5]:
train_df = pd.read_csv(train_path,header=0)
train_df = train_df.rename(columns={"Class Index":"label","Title":"title","Description":"description"})

test_df = pd.read_csv(test_path,header=0)
test_df = test_df.rename(columns={"Class Index":"label","Title":"title","Description":"description"})


# zero-based labels
train_df["label"] = train_df["label"].astype(int) - 1
test_df["label"]  = test_df["label"].astype(int) - 1

# combine title + description
train_df["text"] = train_df["title"] + " " + train_df["description"]
test_df["text"]  = test_df["title"]  + " " + test_df["description"]

In [6]:
# ────────────────────────────────────────────────────────────────────────────────
# 2) Train/validation split
# ────────────────────────────────────────────────────────────────────────────────
train_texts, val_texts, train_labels, val_labels = train_test_split(
    train_df["text"].values,
    train_df["label"].values,
    test_size=0.2,
    random_state=42,
    stratify=train_df["label"].values
)
test_texts  = test_df["text"].values
test_labels = test_df["label"].values


In [7]:
# ────────────────────────────────────────────────────────────────────────────────
# 3) TextVectorization
# ────────────────────────────────────────────────────────────────────────────────
vectorizer = layers.TextVectorization(
    max_tokens=MAX_VOCAB_SIZE,
    output_mode="int",
    output_sequence_length=MAX_SEQUENCE_LEN
)
vectorizer.adapt(train_texts)

def vectorize_text(text, label):
    text = tf.expand_dims(text, -1)
    token_ids = vectorizer(text)
    return tf.squeeze(token_ids, axis=0), label

def make_dataset(texts, labels, shuffle=False):
    ds = tf.data.Dataset.from_tensor_slices((texts, labels))
    if shuffle:
        ds = ds.shuffle(len(texts), seed=42)
    ds = ds.map(vectorize_text, num_parallel_calls=AUTOTUNE)
    return ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)

train_ds = make_dataset(train_texts, train_labels, shuffle=True)
val_ds   = make_dataset(val_texts,   val_labels)
test_ds  = make_dataset(test_texts,  test_labels)

In [8]:
embeddings_index = {}
glovefile = open(GLOVE_FILE,'r',encoding='utf-8')
for line in tqdm(glovefile):
    values = line.split(" ")
    word = values[0]
    coefs = np.asarray(values[1:], dtype='float32')
    embeddings_index[word] = coefs
glovefile.close()

print('Found %s word vectors.' % len(embeddings_index))

1917494it [00:51, 36935.66it/s]

Found 1917494 word vectors.


In [9]:
""" 
embeddings_index["hello"] = array([0.1,0.2,0.3], dtype=float32)
{
  "hello": array([ 0.1,  0.2,  0.3], dtype=float32),
  "world": array([ 0.4,  0.5,  0.6], dtype=float32),
  "test":  array([-0.1, 0.0,  0.1], dtype=float32)
}

"""

' \nembeddings_index["hello"] = array([0.1,0.2,0.3], dtype=float32)\n{\n  "hello": array([ 0.1,  0.2,  0.3], dtype=float32),\n  "world": array([ 0.4,  0.5,  0.6], dtype=float32),\n  "test":  array([-0.1, 0.0,  0.1], dtype=float32)\n}\n\n'

In [10]:
# 1) Build the embedding matrix from your GloVe dict and vectorizer vocab
vocab = vectorizer.get_vocabulary()  # list length ≥ MAX_VOCAB_SIZE
vocab = vocab[:MAX_VOCAB_SIZE]       # truncate to exactly MAX_VOCAB_SIZE
embedding_matrix = np.zeros((MAX_VOCAB_SIZE, EMBEDDING_DIM), dtype="float32")

for idx, word in enumerate(vocab):
    vec = embeddings_index.get(word)
    if vec is not None:
        embedding_matrix[idx] = vec
    # else leave zeros (or add small random noise)

In [11]:
text_inputs = layers.Input(shape=(MAX_SEQUENCE_LEN,),name="input_tokens", dtype="int32")
embedding_layer = layers.Embedding(input_dim=MAX_VOCAB_SIZE, 
                                   output_dim=EMBEDDING_DIM,
                                   input_length=MAX_SEQUENCE_LEN, 
                                   weights=[embedding_matrix], 
                                   trainable=False,
                                   mask_zero =True)
embedded_sequence = embedding_layer(text_inputs)

# 3) Build the rest of the BiLSTM model
lstm1 = layers.Bidirectional(layers.LSTM(LSTM_UNITS, return_sequences=True))(embedded_sequence)
lstm2 = layers.Bidirectional(layers.LSTM(LSTM_UNITS))(lstm1)
x     = layers.Dropout(0.5)(lstm2)
x     = layers.Dense(64, activation="relu")(x)
x     = layers.Dropout(0.5)(x)
out   = layers.Dense(NUM_CLASSES, activation="softmax")(x)

model = Model(inputs=text_inputs, outputs=out, name="bilstm_glove")


/Users/sameerkhan/Desktop/sameerkhan/venv/lib/python3.9/site-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [12]:

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)
model.summary()

Model: "bilstm_glove"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_tokens        │ (None, 200)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, 200, 300)  │  6,000,000 │ input_tokens[0][… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal           │ (None, 200)       │          0 │ input_tokens[0][… │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bidirectional       │ (None, 200, 128)  │    186,880 │ embedding[0][0],  │
│ (Bidirectional)     │                   │            │ not_equal[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bidirectional_1     │ (None, 128)       │     98,816 │ bidirectional[0]… │
│ (Bidirectional)     │                   │            │ not_equal[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 128)       │          0 │ bidirectional_1[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 64)        │      8,256 │ dropout[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 64)        │          0 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 4)         │        260 │ dropout_1[0][0]   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 6,294,212 (24.01 MB)

 Trainable params: 294,212 (1.12 MB)

 Non-trainable params: 6,000,000 (22.89 MB)

In [ ]:
# ────────────────────────────────────────────────────────────────────────────────
# 5) Train
# ────────────────────────────────────────────────────────────────────────────────
ckpt = callbacks.ModelCheckpoint(
    filepath=os.path.join(MODEL_DIR, CHECKPOINT_FILE),
    monitor="val_accuracy",
    save_best_only=True
)
es = callbacks.EarlyStopping(
    monitor="val_loss",
    patience=2,
    restore_best_weights=True
)
model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=[ckpt, es]
)

1339/1500 ━━━━━━━━━━━━━━━━━━━━ 1:19 491ms/step - accuracy: 0.8406 - loss: 0.4593

In [ ]:

# ────────────────────────────────────────────────────────────────────────────────
# 6) Evaluate
# ────────────────────────────────────────────────────────────────────────────────
loss, acc = model.evaluate(test_ds)
print(f"Test accuracy: {acc:.4f}")

119/119 [==============================] - 50s 420ms/step - loss: 0.2727 - accuracy: 0.9061
Test accuracy: 0.9061
